# QuantAlpha AI — Step 3B: Finding a Signal With Real Edge

The simple "RSI 40-65 + ADX>25" signal showed almost no edge (52.1% vs 51.4% baseline — noise,
not a real signal). That's a useful, honest finding: generic textbook indicator combos are heavily
arbitraged and rarely work alone.

This notebook tests **5 improved signal variants** side by side, across **multiple holding
periods**, so we find one with a real, meaningful edge (5+ percentage points above baseline)
before building anything else on top of it.

**Variants tested:**
1. Baseline (original): RSI 40-65 + ADX>25
2. Trend-confirmed: above 50-day AND 200-day SMA (genuine uptrend, not just oscillator reading)
3. Volume-confirmed breakout: price near 20-day high + volume > 1.5x average
4. Combined fundamental + technical: good technical setup AND Piotroski F-Score >= 7
5. Pullback-in-uptrend: price above 200-day SMA but RSI dipped below 40 (buy the dip in a real uptrend)


## 1. Connect to Drive + load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
fscore_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['piotroski_f_score']))

technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)
print(f"Found {len(symbols)} stocks with technical data")
print(f"Found {len(fscore_lookup)} stocks with fundamental scores")


Mounted at /content/drive
Found 200 stocks with technical data
Found 200 stocks with fundamental scores


## 2. Define all signal variants
Each function takes a row of indicator data (at a point in time) and returns True/False —
whether that signal fires on that date for that stock.


In [2]:
def signal_baseline(df, i):
    rsi = df.loc[i, 'RSI_14']
    adx = df.loc[i, 'ADX_14']
    if pd.isna(rsi) or pd.isna(adx):
        return None
    return (40 <= rsi <= 65) and (adx > 25)

def signal_trend_confirmed(df, i):
    close = df.loc[i, 'Close']
    sma50 = df.loc[i, 'SMA_50']
    sma200 = df.loc[i, 'SMA_200']
    rsi = df.loc[i, 'RSI_14']
    if pd.isna(sma50) or pd.isna(sma200) or pd.isna(rsi):
        return None
    return (close > sma50) and (close > sma200) and (40 <= rsi <= 70)

def signal_volume_breakout(df, i, lookback=20):
    if i < lookback:
        return None
    close = df.loc[i, 'Close']
    recent_high = df.loc[i-lookback:i, 'Close'].max()
    volume = df.loc[i, 'Volume']
    avg_volume = df.loc[i-lookback:i, 'Volume'].mean()
    if pd.isna(volume) or pd.isna(avg_volume) or avg_volume == 0:
        return None
    near_high = close >= recent_high * 0.98
    volume_spike = volume > avg_volume * 1.5
    return near_high and volume_spike

def signal_fundamental_technical_combo(df, i, symbol, fscore_lookup):
    rsi = df.loc[i, 'RSI_14']
    adx = df.loc[i, 'ADX_14']
    if pd.isna(rsi) or pd.isna(adx):
        return None
    fscore = fscore_lookup.get(symbol, np.nan)
    if pd.isna(fscore):
        return None
    good_technical = (40 <= rsi <= 65) and (adx > 25)
    good_fundamental = fscore >= 7
    return good_technical and good_fundamental

def signal_pullback_in_uptrend(df, i):
    close = df.loc[i, 'Close']
    sma200 = df.loc[i, 'SMA_200']
    rsi = df.loc[i, 'RSI_14']
    if pd.isna(sma200) or pd.isna(rsi):
        return None
    in_uptrend = close > sma200
    pulled_back = rsi < 40
    return in_uptrend and pulled_back

SIGNAL_FUNCTIONS = {
    'baseline_rsi_adx': signal_baseline,
    'trend_confirmed': signal_trend_confirmed,
    'volume_breakout': signal_volume_breakout,
    'fundamental_technical_combo': signal_fundamental_technical_combo,
    'pullback_in_uptrend': signal_pullback_in_uptrend,
}
print(f"{len(SIGNAL_FUNCTIONS)} signal variants defined.")


5 signal variants defined.


## 3. Run backtest for all variants × multiple holding periods
This loops through every stock's full history once, evaluating all 5 signals simultaneously at
each date — much faster than re-scanning history 5 separate times.


In [3]:
HOLDING_PERIODS = [5, 10, 15, 20]

def backtest_all_signals(symbol, fscore_lookup):
    df = pd.read_parquet(f"data/technical/{symbol}.parquet")
    if len(df) < 300:
        return []
    df = df.reset_index(drop=True)

    rows = []
    max_hold = max(HOLDING_PERIODS)
    for i in range(200, len(df) - max_hold):
        entry_price = df.loc[i, 'Close']
        signals_fired = {}
        signals_fired['baseline_rsi_adx'] = signal_baseline(df, i)
        signals_fired['trend_confirmed'] = signal_trend_confirmed(df, i)
        signals_fired['volume_breakout'] = signal_volume_breakout(df, i)
        signals_fired['fundamental_technical_combo'] = signal_fundamental_technical_combo(df, i, symbol, fscore_lookup)
        signals_fired['pullback_in_uptrend'] = signal_pullback_in_uptrend(df, i)

        for hold in HOLDING_PERIODS:
            exit_price = df.loc[i + hold, 'Close']
            fwd_return = (exit_price - entry_price) / entry_price
            row = {'holding_period': hold, 'fwd_return': fwd_return}
            row.update(signals_fired)
            rows.append(row)
    return rows

all_rows = []
processed = 0
for sym in symbols:
    rows = backtest_all_signals(sym, fscore_lookup)
    all_rows.extend(rows)
    processed += 1
    if processed % 25 == 0:
        print(f"Progress: {processed}/{len(symbols)} stocks processed, {len(all_rows)} observations so far")

bt = pd.DataFrame(all_rows)
print(f"\nTotal observations: {len(bt)}")


Progress: 25/200 stocks processed, 52388 observations so far
Progress: 50/200 stocks processed, 104776 observations so far
Progress: 75/200 stocks processed, 152984 observations so far
Progress: 100/200 stocks processed, 201460 observations so far
Progress: 125/200 stocks processed, 249664 observations so far
Progress: 150/200 stocks processed, 302036 observations so far
Progress: 175/200 stocks processed, 349832 observations so far
Progress: 200/200 stocks processed, 397396 observations so far

Total observations: 397396


## 4. Results — win rate and edge per signal, per holding period
This is the key output. We're looking for any signal where win rate is meaningfully above
baseline (5+ percentage points) and consistent across holding periods, not a one-off fluke.


In [4]:
results_summary = []

for hold in HOLDING_PERIODS:
    subset = bt[bt['holding_period'] == hold]
    overall_win_rate = (subset['fwd_return'] > 0).mean() * 100
    overall_avg_return = subset['fwd_return'].mean() * 100

    for signal_name in SIGNAL_FUNCTIONS.keys():
        fired = subset[subset[signal_name] == True]
        not_fired = subset[subset[signal_name] == False]
        if len(fired) < 30:  # not enough samples to trust
            continue
        win_rate_fired = (fired['fwd_return'] > 0).mean() * 100
        avg_return_fired = fired['fwd_return'].mean() * 100
        win_rate_not_fired = (not_fired['fwd_return'] > 0).mean() * 100 if len(not_fired) else np.nan

        results_summary.append({
            'signal': signal_name,
            'holding_period': hold,
            'n_signals': len(fired),
            'win_rate_when_fired': round(win_rate_fired, 1),
            'win_rate_baseline': round(win_rate_not_fired, 1),
            'edge_pct_points': round(win_rate_fired - win_rate_not_fired, 1),
            'avg_return_when_fired_pct': round(avg_return_fired, 2)
        })

summary_df = pd.DataFrame(results_summary).sort_values('edge_pct_points', ascending=False)
summary_df.to_csv('data/signal_comparison_backtest.csv', index=False)
print(summary_df.to_string(index=False))


                     signal  holding_period  n_signals  win_rate_when_fired  win_rate_baseline  edge_pct_points  avg_return_when_fired_pct
        pullback_in_uptrend               5       4865                 53.6               50.5              3.1                       0.35
fundamental_technical_combo              10      10717                 52.6               51.1              1.5                       0.59
fundamental_technical_combo              15      10717                 53.1               51.9              1.1                       0.71
           baseline_rsi_adx               5      18485                 51.4               50.5              0.9                       0.27
fundamental_technical_combo               5      10717                 51.4               50.6              0.9                       0.29
            trend_confirmed               5      37028                 51.2               50.3              0.8                       0.25
            trend_confirmed

## 5. How to read this table

- **`edge_pct_points`** is what matters most — this is win-rate-when-signal-fired MINUS baseline
  win rate. Look for **5+ points, ideally consistent across multiple holding periods** for the
  same signal (consistency = more trustworthy, not a fluke of one specific horizon)
- **`n_signals`** — don't trust an edge based on fewer than ~100 observations; could be noise
- A signal that only shows edge at one holding period (e.g., 20 days) but not others is less
  trustworthy than one that shows edge at 10, 15, AND 20 days consistently
- If NONE of these show meaningful edge, that's still valuable information — it means we need to
  either combine signals differently, add sentiment as a real filter, or accept that pure
  technical timing is genuinely hard and the fundamental/long-term angle may be the stronger part
  of this whole platform


## 6. Next step depends on this result
- **If a signal shows real edge (5+ points, consistent):** we rebuild Step 3's scoring engine
  using that signal's logic instead of the generic RSI/ADX combo, then move to Step 4
- **If nothing shows real edge:** that's honest, useful data — we pivot toward Position/Long-Term
  modes being the stronger, more reliable modes for this platform (since fundamentals and cash
  flow quality tend to have more genuine persistence than short-term technical timing), and treat
  Swing mode more conservatively or as "in development"

Send me the printed table above — we'll decide the next concrete step from real numbers, not guesses.
